#### [PREV](2.OOP_Multi_Feature_Linear_Regression.ipynb) | [HOME](../README.md) | [NEXT](4.OOP_Logistic_Regression.ipynb)

---

# OOP Polynomial Linear Regression Implementation

This Jupyter Notebook provides a basic implementation of Polynomial Linear Regression. The data set contains measurements on a single species of insect captured on two continents. We want to predict an insect's wing size given the other measurements we have (continent, latitude and sex).

#### Step 1: Import Dependencies

In [ ]:
import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error, mean_squared_error

#### Step 2: Import And Visualise The Data

In [ ]:
# Import data as a Pandas DataFrame
insects = pd.read_csv('./data/insects.csv', sep='\t')
insects = pd.DataFrame({
    'continent': insects['continent'],
    'latitude': insects['latitude'],
    'sex': insects['sex'],
    'wingsize': insects['wingsize']
})

# Features variable
X_insects = insects[['continent', 'latitude', 'sex']]
# Target variable
y_insects = insects['wingsize']

insects.head()

In [ ]:
# Plot the data, colour coded by sex
plt.scatter(insects.latitude, insects.wingsize, c=insects['sex'])
plt.xlabel("Latitude")
plt.ylabel("Wing size")
plt.colorbar(label="Sex")
plt.show()

#### Step 3: Instantiate a LinearRegression Object

In [ ]:
insects_regression = LinearRegression()

#### Step 4: Fit the model

In [ ]:
# Create polynomial features up to degree 4
poly = PolynomialFeatures(degree=4, include_bias=False)

# Fit and transform the features
poly_features = poly.fit_transform(X_insects)

# Fit the regression model
insects_regression.fit(poly_features, y_insects)

#### Step 5: Predict Values

In [ ]:
# Predict the target for the whole dataset
wing_size_predictions = insects_regression.predict(poly_features)

# Predict the target for a new data point
new_insect = pd.DataFrame({
    'continent': [1],
    'latitude': [40],
    'sex': [0]
})
# Use transform (not fit_transform) so the new data gets the same polynomial features as the training data
new_poly_features = poly.transform(new_insect)
new_insect['wingsize'] = insects_regression.predict(new_poly_features)
print(f"New insect is:\n{new_insect}")

#### Step 6: Plot the Predictions

In [ ]:
# Plot the predictions compared to the actual data
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].scatter(insects.latitude, insects.wingsize, c=insects['sex'], s=40)
axs[0].set_title("Actual Data")
axs[0].set_xlabel("Latitude")
axs[0].set_ylabel("Actual Wing Size")

axs[1].scatter(insects.latitude, wing_size_predictions, c=insects['sex'], s=40)
axs[1].set_title("Predicted Data")
axs[1].set_xlabel("Latitude")
axs[1].set_ylabel("Predicted Wing Size")

plt.show()

#### Step 7: Evaluate The Model

In [ ]:
# Get evaluative data from the model
print(f"Model coefficients: {insects_regression.coef_}")
print(f"Model y intercept: {insects_regression.intercept_}")
print(f"Model score (R squared): {insects_regression.score(poly_features, y_insects)}")
mae = mean_absolute_error(y_true=y_insects, y_pred=wing_size_predictions)
mse = mean_squared_error(y_true=y_insects, y_pred=wing_size_predictions)
print("MAE:", mae)
print("MSE:", mse)

In [ ]:
# Optional: Manually evaluate the loss and cost of the model
model_loss = pd.DataFrame({
    'Target': y_insects,
    'Predicted result': wing_size_predictions.round(2),
    'Loss': (wing_size_predictions - y_insects) ** 2  # Squared error for each insect
})
# Cost = 1/(2m) x the sum of the squared errors (half of the MSE printed above)
model_cost = 1 / (2 * model_loss.shape[0]) * model_loss['Loss'].sum()

print(f"The cost of this model is {model_cost:.5f}")
print(model_loss)